# Tests: `fasterai.misc.conv_decomposer` (source `nbs/misc/conv_decomposer.ipynb`)

In [ ]:
from fastcore.test import *
import torch
import torch.nn as nn
from fasterai.misc.conv_decomposer import *

In [ ]:
from fastcore.test import *

decomposer = Conv_Decomposer()
_m = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.Conv2d(16, 32, 3, padding=1))
_x = torch.randn(2, 3, 8, 8)

# === All methods produce correct output shape ===
for method in ['tucker', 'svd', 'spatial', 'cp']:
    _dec = decomposer.decompose(_m, 0.5, method=method)
    test_eq(_m(_x).shape, _dec(_x).shape)
    assert torch.isfinite(_dec(_x)).all(), f"{method} produced non-finite output"

# === Tucker: 3 layers (1x1, KxK, 1x1) ===
_t = decomposer.decompose(_m, 0.5, method='tucker')
test_eq(len(_t[0]), 3)
test_eq(_t[0][0].kernel_size, (1, 1))
test_eq(_t[0][1].kernel_size, (3, 3))

# === SVD: 2 layers (KxK, 1x1) ===
_s = decomposer.decompose(_m, 0.5, method='svd')
test_eq(len(_s[0]), 2)
test_eq(_s[0][0].kernel_size, (3, 3))
test_eq(_s[0][1].kernel_size, (1, 1))

# === Spatial: 2 layers (Kx1, 1xK) ===
_sp = decomposer.decompose(_m, 0.5, method='spatial')
test_eq(len(_sp[0]), 2)
test_eq(_sp[0][0].kernel_size, (3, 1))
test_eq(_sp[0][1].kernel_size, (1, 3))

# === CP: 4 layers (1x1, Kx1, 1xK, 1x1) ===
_cp = decomposer.decompose(_m, 0.5, method='cp')
test_eq(len(_cp[0]), 4)
test_eq(_cp[0][0].kernel_size, (1, 1))  # pointwise in
test_eq(_cp[0][1].kernel_size, (3, 1))  # depthwise vertical
test_eq(_cp[0][2].kernel_size, (1, 3))  # depthwise horizontal
test_eq(_cp[0][3].kernel_size, (1, 1))  # pointwise out

# === Common: 1x1 and grouped skipped ===
assert isinstance(decomposer.decompose(nn.Sequential(nn.Conv2d(16, 32, 1)), 0.5)[0], nn.Conv2d)
assert isinstance(decomposer.decompose(nn.Sequential(nn.Conv2d(16, 16, 3, groups=16, padding=1)), 0.5)[0], nn.Conv2d)

# === Bias: last layer gets it ===
for method in ['tucker', 'svd', 'spatial', 'cp']:
    _dec = decomposer.decompose(nn.Sequential(nn.Conv2d(16, 32, 3, padding=1, bias=True)), 0.5, method=method)
    seq = _dec[0]
    assert seq[-1].bias is not None, f"{method}: last layer missing bias"
    for layer in seq[:-1]:
        assert layer.bias is None, f"{method}: non-last layer has bias"

# === Stride transfer ===
_stride = decomposer.Tucker(nn.Conv2d(16, 32, 3, stride=2, padding=1), 0.5)
test_eq(_stride[1].stride, (2, 2))

_svd_stride = decomposer.SVD(nn.Conv2d(16, 32, 3, stride=2, padding=1), 0.5)
test_eq(_svd_stride[0].stride, (2, 2))

# === Validation ===
with ExceptionExpected(ValueError): decomposer.decompose(nn.Sequential(nn.Conv2d(3, 16, 3)), percent_removed=1.0)
with ExceptionExpected(ValueError): decomposer.decompose(nn.Sequential(nn.Conv2d(3, 16, 3)), method='bad')

# === energy_threshold + layers/exclude ===
_m4 = nn.Sequential(nn.Conv2d(16, 32, 3, padding=1))
assert decomposer.decompose(_m4, energy_threshold=0.99)[0][0].out_channels >= \
       decomposer.decompose(_m4, 0.5)[0][0].out_channels

_m5 = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.Conv2d(16, 32, 3, padding=1))
assert isinstance(decomposer.decompose(_m5, 0.5, layers=['0'])[2], nn.Conv2d)
assert isinstance(decomposer.decompose(_m5, 0.5, exclude=['2'])[2], nn.Conv2d)

In [ ]:
#| slow
from torchvision.models import resnet18

# Decompose a real ResNet-18, verify it still works
_resnet = resnet18(weights=None)
_resnet.eval()
_x = torch.randn(2, 3, 64, 64)
_out_orig = _resnet(_x)

_dec = Conv_Decomposer()
_resnet_dec = _dec.decompose(_resnet, percent_removed=0.5)
_resnet_dec.eval()
_out_dec = _resnet_dec(_x)

# Same output shape
test_eq(_out_orig.shape, _out_dec.shape)

# Outputs are finite (no NaN/Inf)
assert torch.isfinite(_out_dec).all(), "Decomposed ResNet produced non-finite outputs"

# Parameter count reduced
_orig_params = sum(p.numel() for p in _resnet.parameters())
_dec_params = sum(p.numel() for p in _resnet_dec.parameters())
assert _dec_params < _orig_params, f"Expected fewer params: {_dec_params} >= {_orig_params}"
print(f"ResNet-18: {_orig_params:,} → {_dec_params:,} params ({_orig_params/_dec_params:.2f}x compression)")